# 🤖 Baustein 4 — Entscheidungsbaum (Decision Tree)

**IHK-Themen abgedeckt:** Decision Tree, Metriken (Accuracy/Precision/Recall/F1/Confusion Matrix), R², OOP, joblib

**Basis:** `analysetabelle.csv` (1.824 Krankenhäuser, 15 Spalten)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
from pathlib import Path

# ── Projekt-Root sicherstellen (Notebook liegt in /Notebooks) ────
if Path.cwd().name == 'Notebooks':
    os.chdir(Path.cwd().parent)
print(f'Arbeitsverzeichnis: {Path.cwd()}')

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    r2_score
)
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("Data/analysetabelle.csv", low_memory=False)

print(f'Datensatz: {df.shape}')
df.head(3)

Arbeitsverzeichnis: C:\Users\funke\OneDrive\PhytonDataCraftKurs\QualitaetsMusterFinderProjekt
Datensatz: (1824, 18)


---
## 1️⃣ OOP — Modell-Pipeline als Klasse

In [3]:
import sys
sys.path.insert(0, str(Path.cwd() / "scripts"))
from modell_klasse import KrankenhausModell

print("KrankenhausModell importiert aus scripts/modell_klasse.py ✓")
print(f"FEATURE_COLS: {KrankenhausModell.FEATURE_COLS}")

KrankenhausModell importiert aus scripts/modell_klasse.py ✓
FEATURE_COLS: ['SO.Betten', 'SO.Uni', 'fortbildungsquote', 'aerzte_pro_bett', 'pflege_pro_bett', 'ist_konzern']


---
## 2️⃣ Train-Test-Split & Basislinie

In [5]:
modell = KrankenhausModell(max_depth=3, random_state=42)
X, y = modell.prepare(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training:  {len(X_train):,} Häuser  ({len(X_train)/len(X):.0%})')
print(f'Test:      {len(X_test):,} Häuser  ({len(X_test)/len(X):.0%})')
print(f'\nFeatures: {modell.feature_names}')

# Basislinie: immer die häufigste Klasse vorhersagen
baseline = y_test.value_counts(normalize=True).max()
print(f'\nBasislinie (häufigste Klasse):  {baseline:.1%}')
print('→ Das Modell muss besser als die Basislinie sein.')

Training:  1,459 Häuser  (80%)
Test:      365 Häuser  (20%)

Features: ['SO.Betten', 'SO.Uni', 'fortbildungsquote', 'aerzte_pro_bett', 'pflege_pro_bett', 'ist_konzern', 'traeger_enc']

Basislinie (häufigste Klasse):  50.7%
→ Das Modell muss besser als die Basislinie sein.


---
## 3️⃣ Decision Tree trainieren & bewerten

In [7]:
modell.fit(X_train, y_train)
ergebnis = modell.evaluate(X_test, y_test)

print('── Metriken auf Testdaten ──')
print(f"  Accuracy:  {ergebnis['accuracy']:.3f}")
print(f"  Precision: {ergebnis['precision']:.3f}  (Wie viele der 'viele Probleme'-Vorhersagen stimmen?)")
print(f"  Recall:    {ergebnis['recall']:.3f}   (Wie viele echten 'viele Probleme' wurden gefunden?)")
print(f"  F1-Score:  {ergebnis['f1']:.3f}")
print(f"  Basislinie:{baseline:.3f}")
print(f"  Verbesserung über Basislinie: {ergebnis['accuracy']-baseline:+.3f}")

# Cross-Validation
cv_scores = cross_val_score(modell.model, X, y, cv=5, scoring='accuracy')
print(f'\n  5-Fold CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

── Metriken auf Testdaten ──
  Accuracy:  0.636
  Precision: 0.682  (Wie viele der 'viele Probleme'-Vorhersagen stimmen?)
  Recall:    0.489   (Wie viele echten 'viele Probleme' wurden gefunden?)
  F1-Score:  0.570
  Basislinie:0.507
  Verbesserung über Basislinie: +0.129

  5-Fold CV Accuracy: 0.597 ± 0.042


In [8]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=ergebnis['cm'],
    display_labels=['Wenige Probleme', 'Viele Probleme']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Decision Tree (max_depth=3)', fontsize=12)
plt.tight_layout()
plt.savefig('grafiken/confusion_matrix.png', dpi=120)
plt.show()
print('Grafik gespeichert: grafiken/confusion_matrix.png')

Grafik gespeichert: grafiken/confusion_matrix.png


---
## 4️⃣ Entscheidungsbaum visualisieren

In [10]:
fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(
    modell.model,
    feature_names=modell.feature_names,
    class_names=['Wenige Probleme', 'Viele Probleme'],
    filled=True, rounded=True, fontsize=9, ax=ax
)
ax.set_title('Entscheidungsbaum — max_depth=3', fontsize=13)
plt.tight_layout()
plt.savefig('grafiken/decision_tree.png', dpi=120)
plt.show()
print('Grafik gespeichert: grafiken/decision_tree.png')

Grafik gespeichert: grafiken/decision_tree.png


In [11]:
# Feature Importance
importance = pd.Series(
    modell.model.feature_importances_,
    index=modell.feature_names
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
importance.plot(kind='barh', ax=ax, color='#2980b9', edgecolor='white')
ax.set_title('Feature Importance — Decision Tree', fontsize=12)
ax.set_xlabel('Wichtigkeit')
plt.tight_layout()
plt.savefig('grafiken/feature_importance.png', dpi=120)
plt.show()

print('\nFeature Importance:')
for feat, imp in importance.sort_values(ascending=False).items():
    print(f'  {feat:25s}: {imp:.4f}')


Feature Importance:
  aerzte_pro_bett          : 0.5355
  pflege_pro_bett          : 0.2384
  SO.Betten                : 0.2261
  traeger_enc              : 0.0000
  fortbildungsquote        : 0.0000
  SO.Uni                   : 0.0000
  ist_konzern              : 0.0000


---
## 5️⃣ R²-Metrik — Lineare Regression zum Vergleich

In [13]:
# R² erklaert: wie viel Varianz der Ziel-Variable erklaert das Modell?
# Wert 0 = kein Erklarungswert, 1 = perfekte Vorhersage
# Fuer Klassifikation: Decision Tree Accuracy ist das Pendant zu R²

# Lineare Regression auf auffaellig_quote (kontinuierlich) -> R² zeigen
lr = LinearRegression()
lr.fit(X_train, y_train)  # y_train ist 0/1, aber LR funktioniert auch
y_pred_lr = lr.predict(X_test)
r2 = r2_score(y_test, y_pred_lr)

# Besser: R² auf der kontinuierlichen auffaellig_quote
X2, y2 = modell.prepare(df)
y_cont = df['auffaellig_quote']  # kontinuierliche Zielvariable
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y_cont, test_size=0.2, random_state=42)
lr2 = LinearRegression().fit(X2_train, y2_train)
r2_cont = r2_score(y2_test, lr2.predict(X2_test))

print('── R²-Metrik (Bestimmtheitsmaß) ──')
print(f'  R² (Lineare Regression auf Ziel-Variable 0/1):  {r2:.4f}')
print(f'  R² (Lineare Regression auf auffaellig_quote):   {r2_cont:.4f}')
print()
print('Interpretation:')
print(f'  R² = {r2_cont:.2f} bedeutet: Das Modell erklaert nur {r2_cont*100:.1f}% der Varianz.')
print('  Das bestaetigt den Befund der deskriptiven Analyse:')
print('  Strukturmerkmale erklaeren die Auffaelligkeit nur sehr schwach.')
print('  → Kein Zusammenhang ist ein valides Ergebnis!')

# Koeffizienten
print('\nLineare Regression Koeffizienten (auffaellig_quote):')
for feat, coef in zip(modell.feature_names, lr2.coef_):
    print(f'  {feat:25s}: {coef:+.5f}')
print(f'  Intercept: {lr2.intercept_:.4f}')

── R²-Metrik (Bestimmtheitsmaß) ──
  R² (Lineare Regression auf Ziel-Variable 0/1):  0.0086
  R² (Lineare Regression auf auffaellig_quote):   0.0329

Interpretation:
  R² = 0.03 bedeutet: Das Modell erklaert nur 3.3% der Varianz.
  Das bestaetigt den Befund der deskriptiven Analyse:
  Strukturmerkmale erklaeren die Auffaelligkeit nur sehr schwach.
  → Kein Zusammenhang ist ein valides Ergebnis!

Lineare Regression Koeffizienten (auffaellig_quote):
  SO.Betten                : -0.00003
  SO.Uni                   : -0.00778
  fortbildungsquote        : -0.01978
  aerzte_pro_bett          : -0.04087
  pflege_pro_bett          : -0.04094
  ist_konzern              : +0.00125
  traeger_enc              : +0.00298
  Intercept: 0.8358


---
## 6️⃣ Modell speichern & laden (joblib)

In [15]:
# Modell speichern (Default-Pfad aus modell_klasse.py -> Data/modell_krankenhaus.pkl)
modell.save()

# Modell laden und Probe-Vorhersage
geladenes_modell = KrankenhausModell.load()

# Beispiel: Ein Krankenhaus mit bestimmten Merkmalen
beispiel = pd.DataFrame([{
    'SO.Betten': 300, 'SO.Uni': 0,
    'fortbildungsquote': 0.8, 'aerzte_pro_bett': 0.4,
    'pflege_pro_bett': 1.0, 'ist_konzern': 0,
    'traeger_enc': 0  # privat
}])
vorhersage = geladenes_modell.model.predict(beispiel)[0]
proba = geladenes_modell.model.predict_proba(beispiel)[0]

print(f'Beispiel-Vorhersage: {"Viele Probleme" if vorhersage==1 else "Wenige Probleme"}')
print(f'Wahrscheinlichkeit:  Wenige={proba[0]:.2%}  |  Viele={proba[1]:.2%}')
print()
print('Modell geladen und einsatzbereit ✓')


Modell gespeichert: C:\Users\funke\OneDrive\PhytonDataCraftKurs\QualitaetsMusterFinderProjekt\Data\modell_krankenhaus.pkl
Beispiel-Vorhersage: Wenige Probleme
Wahrscheinlichkeit:  Wenige=59.71%  |  Viele=40.29%

Modell geladen und einsatzbereit ✓


---
## 7️⃣ Zusammenfassung Baustein 4

| Metrik | Wert | Interpretation |
|--------|------|----------------|
| Accuracy | siehe oben | Anteil korrekt klassifizierter Häuser |
| Precision | siehe oben | Wie verlässlich ist eine 'Viele Probleme'-Vorhersage? |
| Recall | siehe oben | Wie viele echte Problemhäuser werden erkannt? |
| F1-Score | siehe oben | Harmonsches Mittel aus Precision & Recall |
| R² | sehr niedrig | Strukturmerkmale erklären Qualitätsprobleme kaum |

> ⚠️ **Kein Zusammenhang ist ein valides Ergebnis.** Ein niedriger R²-Wert bestätigt die Befunde aus Baustein 2.